# Statistical Inference: Worked Solutions

In [ ]:
import numpy as np
from statistics import NormalDist
NORMAL = NormalDist()
Z95 = NORMAL.inv_cdf(0.975)


## Problem 1. A Bernoulli likelihood and empirical variance

**Problem.** Compute the Bernoulli maximum likelihood estimate for a nonempty binary sample, including all-zero and all-one samples. Verify it against a likelihood grid. For $(0,0,1,2)$, compare the plug-in and unbiased variance estimates.

**Solution.** With $s=\sum_i x_i$ successes among $n$ observations, the log likelihood is $s\log p+(n-s)\log(1-p)$. Its interior derivative vanishes at $\widehat p=s/n$; for $s=0$ or $s=n$, the maximum over $[0,1]$ is the corresponding endpoint. The displayed ten-trial sample has seven successes, so $\widehat p=0.7$.

For the four real observations, $\overline x=3/4$ and $\sum_i(x_i-\overline x)^2=11/4$. Dividing by $n$ gives plug-in variance $11/16$; dividing by $n-1$ gives the unbiased population-variance estimate $11/12$ under iid sampling with finite variance.

In [ ]:
def bernoulli_mle(observations):
    x = np.asarray(observations, dtype=float)
    if x.ndim != 1 or x.size == 0 or not np.all((x == 0) | (x == 1)):
        raise ValueError("nonempty binary sample required")
    return float(x.mean())


In [ ]:
x = np.array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1])
p_hat = bernoulli_mle(x)
np.testing.assert_allclose(p_hat, 0.7)
np.testing.assert_allclose(bernoulli_mle([0, 0, 0]), 0)
np.testing.assert_allclose(bernoulli_mle([1, 1]), 1)
grid = np.linspace(0.01, 0.99, 99)
loglikelihood = x.sum() * np.log(grid) + (x.size - x.sum()) * np.log1p(-grid)
np.testing.assert_allclose(grid[np.argmax(loglikelihood)], p_hat)
# The sample variance illustrates different estimation criteria.
sample = np.array([0., 0., 1., 2.])
np.testing.assert_allclose(sample.var(ddof=0), 11 / 16)
np.testing.assert_allclose(sample.var(ddof=1), 11 / 12)
print({"mle": p_hat, "plug_in_variance": sample.var(), "unbiased_variance": sample.var(ddof=1)})


## Problem 2. Confidence intervals, coverage, and estimator MSE

**Problem.** Construct a two-sided Gaussian mean interval when the standard deviation $\sigma$ is known. Simulate its coverage and width at $n=25$ and $n=100$. In a separate Bernoulli experiment with ten trials, compare the sample proportion with the shrinkage estimator $T=0.9\widehat p+0.1(0.5)$ at $p=0.1$ and $p=0.01$.

**Solution.** Under iid Gaussian sampling, $(\overline X-\mu)/(\sigma/\sqrt n)$ is standard Gaussian. For its $0.975$ quantile $z$, the interval is

$$
\left[\overline X-z\frac{\sigma}{\sqrt n},\ \overline X+z\frac{\sigma}{\sqrt n}\right].
$$

Its exact coverage is $0.95$ under this model. Quadrupling $n$ halves its width. Repeated datasets estimate coverage while holding the population mean fixed; replacing known $\sigma$ by an estimate requires a different justification.

For the shrinkage calculation,

$$
\operatorname{MSE}(T)=0.9^2\frac{p(1-p)}{10}+[0.1(0.5-p)]^2.
$$

At $p=0.1$ this is $0.00889$, below the sample proportion's MSE $0.009$. At $p=0.01$ it is $0.0032029$, above $0.00099$. The same shrinkage rule can help near its target and hurt farther away. The simulations check these population calculations.

In [ ]:
def known_sigma_mean_interval(observations, sigma, z=Z95):
    x = np.asarray(observations, dtype=float)
    if x.ndim != 1 or x.size == 0 or not np.all(np.isfinite(x)):
        raise ValueError("nonempty finite sample required")
    if not np.isfinite(sigma) or sigma <= 0 or not np.isfinite(z) or z <= 0:
        raise ValueError("positive finite scale and quantile required")
    width = z * sigma / np.sqrt(x.size)
    return float(x.mean() - width), float(x.mean() + width)


In [ ]:
lo, hi = known_sigma_mean_interval(np.full(25, 1.2), 2.0, z=1.96)
np.testing.assert_allclose([lo, hi], [0.416, 1.984])
np.testing.assert_allclose(known_sigma_mean_interval([3.0], 1.0, z=2.0), [1.0, 5.0])
rng = np.random.default_rng(8108)
mu, sigma, repetitions = 1.2, 2.0, 12000
results = []
for n in (25, 100):
    data = rng.normal(mu, sigma, size=(repetitions, n))
    means = data.mean(axis=1)
    intervals = np.array([known_sigma_mean_interval(row, sigma) for row in data])
    coverage = np.mean((intervals[:, 0] <= mu) & (mu <= intervals[:, 1]))
    assert abs(coverage - 0.95) < 0.012
    assert abs(means.var() / (sigma ** 2 / n) - 1) < 0.06
    results.append((n, coverage, np.mean(intervals[:, 1] - intervals[:, 0])))
np.testing.assert_allclose(results[0][2] / results[1][2], 2.0)
print("n, empirical coverage, interval width")
for row in results:
    print(row)
# MSE can be estimated over repeated datasets, with the generating p fixed.
rng = np.random.default_rng(8109)
for p in (0.1, 0.01):
    means = rng.binomial(10, p, size=200000) / 10
    shrink = 0.9 * means + 0.1 * 0.5
    theory = 0.9 ** 2 * p * (1 - p) / 10 + (0.1 * (0.5 - p)) ** 2
    assert abs(np.mean((shrink - p) ** 2) - theory) < 0.0001
    print({"p": p, "sample_mean_MSE": np.mean((means - p) ** 2), "shrinkage_MSE": theory})


## Problem 3. Bootstrap means from one observed sample

**Problem.** For the fixed dataset $x=(0,0,1,2)$, draw bootstrap samples of the same size with replacement and return their means. Check their conditional mean and variance, estimate their standard deviation, and calculate a percentile interval. Verify reproducibility for a fixed seed and the behavior of a constant dataset.

**Solution.** Each bootstrap observation is drawn from the empirical law, which places masses $1/2,1/4,1/4$ at $0,1,2$. Drawing indices uniformly with replacement implements this law. With the original data held fixed,

$$
E^*[\overline X^*]=\overline x=\frac34,\qquad
\operatorname{Var}^*(\overline X^*)=\frac{1}{n}\left[\frac1n\sum_i(x_i-\overline x)^2\right]=\frac{11}{64}.
$$

Thus the exact conditional standard deviation is $\sqrt{11}/8\approx0.4146$. The simulated standard deviation approximates this value. Empirical $2.5\%$ and $97.5\%$ quantiles define the percentile interval; they need not give exact $95\%$ population coverage, especially for such a small sample. A constant empirical law gives a constant bootstrap mean. More resamples improve the Monte Carlo approximation without adding population observations.

In [ ]:
def bootstrap_means(observations, repetitions, seed):
    x = np.asarray(observations, dtype=float)
    if x.ndim != 1 or x.size < 2 or not np.all(np.isfinite(x)):
        raise ValueError("at least two finite observations required")
    if not isinstance(repetitions, int) or repetitions < 2:
        raise ValueError("at least two bootstrap repetitions required")
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, x.size, size=(repetitions, x.size))
    return x[indices].mean(axis=1)


In [ ]:
fixed_sample = np.array([0.0, 0.0, 1.0, 2.0])
replicates = bootstrap_means(fixed_sample, 40000, 8110)
assert replicates.shape == (40000,)
np.testing.assert_array_equal(replicates, bootstrap_means(fixed_sample, 40000, 8110))
assert abs(replicates.mean() - fixed_sample.mean()) < 0.01
assert abs(replicates.var() - fixed_sample.var() / fixed_sample.size) < 0.006
np.testing.assert_array_equal(bootstrap_means([2, 2, 2], 5, 1), np.full(5, 2.0))
print({"bootstrap_SE": replicates.std(ddof=1), "exact_conditional_SE": np.sqrt(fixed_sample.var() / 4), "percentile_interval": np.quantile(replicates, [0.025, 0.975])})


## Problem 4. A beta posterior and a calibrated Gaussian test

**Problem.** Update a $\operatorname{Beta}(2,2)$ prior using the ten-trial Bernoulli sample from Problem 1. Compute posterior moments, its mode, the next-trial success probability, and an approximate $95\%$ credible interval using posterior draws. Separately, check a two-sided Gaussian test's rejection rate when its null hypothesis holds.

**Solution.** Multiplying the beta prior density by the Bernoulli likelihood adds successes and failures to its two shape parameters. Seven successes and three failures give $\operatorname{Beta}(9,5)$, whose mean, variance, and mode are

$$
\frac9{14},\qquad \frac{9\cdot5}{14^2\cdot15},\qquad \frac{9-1}{9+5-2}=\frac23.
$$

Averaging the next-trial success probability over this posterior gives $9/14$. Posterior quantiles give a credible interval conditional on this dataset and prior. Posterior simulation varies $p$ with the data fixed; the earlier coverage experiment held its generating parameter fixed and varied datasets. With no observations, the update leaves the prior unchanged.

For the final test, the null statistic is $Z\sim\mathcal N(0,1)$ and its two-sided $p$-value is $2[1-\Phi(|Z|)]$. Rejecting when this is below $0.05$ has null probability $0.05$. The simulation checks that probability and does not assign a posterior probability to the null hypothesis.

In [ ]:
def beta_posterior(observations, a, b):
    x = np.asarray(observations, dtype=float)
    if x.ndim != 1 or not np.all((x == 0) | (x == 1)):
        raise ValueError("binary sample required")
    if not np.isfinite(a) or not np.isfinite(b) or a <= 0 or b <= 0:
        raise ValueError("positive finite prior parameters required")
    successes = float(x.sum())
    return a + successes, b + x.size - successes


In [ ]:
a_post, b_post = beta_posterior(x, 2.0, 2.0)
np.testing.assert_allclose([a_post, b_post], [9.0, 5.0])
np.testing.assert_allclose(beta_posterior([], 2.0, 3.0), [2.0, 3.0])
mean = a_post / (a_post + b_post)
variance = a_post * b_post / ((a_post + b_post) ** 2 * (a_post + b_post + 1))
rng = np.random.default_rng(8111)
posterior_draws = rng.beta(a_post, b_post, size=80000)
assert abs(posterior_draws.mean() - mean) < 0.003
assert abs(posterior_draws.var() - variance) < 0.0004
credible_interval = np.quantile(posterior_draws, [0.025, 0.975])
assert 0 < credible_interval[0] < mean < credible_interval[1] < 1
print({"posterior_mean_and_next_success_probability": mean, "posterior_mode": (a_post - 1) / (a_post + b_post - 2), "approximate_95pct_credible_interval": credible_interval})
# Null calibration of a known-variance Gaussian mean test.
z = np.random.default_rng(8112).normal(size=50000)
p_values = np.array([2 * (1 - NORMAL.cdf(abs(value))) for value in z])
assert abs(np.mean(p_values < 0.05) - 0.05) < 0.004
print({"null_rejection_fraction": np.mean(p_values < 0.05)})


## Further problems: worked answers

**1. Identify the estimand, estimator, and numerical estimate in the Bernoulli example. Which object varies in repeated sampling?**

The estimand is the population success probability $p$. The estimator is the random sample proportion $\widehat p=n^{-1}\sum_iX_i$. The realized estimate is $0.7$. In repeated sampling at a fixed $p$, the dataset and estimator vary; $p$ stays fixed.

**2. Why does a prediction interval for one future observation retain positive width as $n$ grows?**

For an independent future Gaussian observation, $\operatorname{Var}(X_{\mathrm{new}}-\overline X)=\sigma^2(1+1/n)$. The uncertainty in estimating the mean vanishes, but the future observation's variance $\sigma^2$ remains. Thus the full width tends to $2z_{0.975}\sigma$, whereas the mean confidence interval's width tends to zero.

**3. What distribution is held fixed in the bootstrap experiment, and how does it differ from the coverage experiment?**

The bootstrap holds the observed empirical distribution $\widehat P_n$ fixed and samples from it. The coverage experiment samples from a specified population law $P_\theta$ with its parameter fixed; each repetition produces a new original dataset. Bootstrap resampling approximates unknown population sampling using one dataset and does not recover information absent from it.

**4. Why can a posterior credible interval and a frequentist confidence interval have similar endpoints yet different meanings?**

A credible interval contains a stated amount of posterior probability, conditional on the observed data, model, and prior. A confidence interval is the realized output of a procedure whose coverage is evaluated over repeated datasets at a fixed parameter. Endpoint agreement does not change which object is random or which assumptions justify the statement; Bayesian credibility need not imply exact frequentist coverage.

**5. Why does a 5% rejection rate under a simulated null not make an observed $p$-value of 0.05 a 5% posterior probability of the null?**

Null calibration describes probabilities of data or decisions *assuming the null is true*. An observed $p$-value is the null probability of a statistic at least as extreme as the observed one, using the specified ordering. A posterior probability of the null conditions in the reverse direction and would require a prior on hypotheses together with likelihoods under the alternatives. Neither is supplied by the null simulation.